In [ ]:
# ============================================================
# BA3002 - BUSINESS DATA AND TEXT MINING
# WEEK 3 PRACTICAL
# WEB SCRAPING DEMO
# Business Case: BookWise Competitor Monitoring
# ============================================================


# ------------------------------------------------------------
# STEP 0: IMPORT REQUIRED LIBRARIES
# ------------------------------------------------------------

import requests                          # Used to send HTTP requests to websites.
from bs4 import BeautifulSoup            # Used to read, parse, and search HTML.
import pandas as pd                      # Used to create and analyze tabular data.


# ------------------------------------------------------------
# STEP 1: WEBSITE
# ------------------------------------------------------------

url = "https://books.toscrape.com/"      # Store the website address in a variable.


# ------------------------------------------------------------
# STEP 2: REQUEST THE WEBSITE
# ------------------------------------------------------------

response = requests.get(url, timeout=10) # Send a GET request to the website with a 10-second timeout.

response.encoding = "utf-8"              # Force UTF-8 encoding to correctly read symbols such as £.

print("Status Code:", response.status_code)  # Display the HTTP status code returned by the server.


# ------------------------------------------------------------
# STEP 3: CHECK WHETHER THE REQUEST WAS SUCCESSFUL
# ------------------------------------------------------------

if response.status_code == 200:          # Check whether the request succeeded.
    print("Website successfully retrieved.") # Inform the user that the webpage was downloaded.
else:                                    # Run this part if the website did not return status code 200.
    print("Website request failed.")      # Display a failure message.


Status Code: 200
Website successfully retrieved.


In [ ]:
# ------------------------------------------------------------
# STEP 4: PARSE THE HTML
# ------------------------------------------------------------

soup = BeautifulSoup(                    # Create a BeautifulSoup object from the webpage.
    response.text,                       # Use the HTML text received from the website.
    "html.parser"                        # Tell BeautifulSoup to use Python's HTML parser.
)

print("Page title:", soup.title.string)  # Display the title of the webpage.

Page title: 
    All products | Books to Scrape - Sandbox



In [ ]:

# ------------------------------------------------------------
# STEP 5: FIND ALL BOOK / PRODUCT CONTAINERS
# ------------------------------------------------------------

books = soup.find_all(                   # Search the webpage for all matching HTML elements.
    "article",                           # Look for <article> HTML tags.
    class_="product_pod"                 # Select only articles with class="product_pod".
)

print("Books Found:", len(books))         # Display how many book containers were found.

Books Found: 20


In [ ]:

# ------------------------------------------------------------
# STEP 6: CREATE RATING MAPPING
# ------------------------------------------------------------

rating_map = {                           # Create a dictionary to convert rating words into numbers.
    "One": 1,                            # Convert One to numeric rating 1.
    "Two": 2,                            # Convert Two to numeric rating 2.
    "Three": 3,                          # Convert Three to numeric rating 3.
    "Four": 4,                           # Convert Four to numeric rating 4.
    "Five": 5                            # Convert Five to numeric rating 5.
}


# ------------------------------------------------------------
# STEP 7: CREATE EMPTY LISTS
# ------------------------------------------------------------

titles = []                              # Empty list for storing book titles.
prices = []                              # Empty list for storing book prices.
ratings = []                             # Empty list for storing book ratings.
availability_list = []                   # Empty list for storing availability information.


# ------------------------------------------------------------
# STEP 8: EXTRACT DATA FROM EVERY BOOK
# ------------------------------------------------------------

for book in books:                       # Loop through every book container found on the webpage.

    title = book.h3.a["title"]            # Extract the full book title from the title attribute.

    price = book.find(                   # Search inside the current book for its price.
        "p",                             # Look for a <p> HTML element.
        class_="price_color"             # Select the paragraph with class="price_color".
    ).get_text(strip=True)               # Extract only the text and remove extra spaces.

    availability = book.find(            # Search inside the current book for availability.
        "p",                             # Look for a <p> HTML element.
        class_="instock availability"    # Select the availability paragraph.
    ).get_text(strip=True)               # Extract availability text and remove extra spaces.

    rating_classes = book.find(          # Find the HTML element containing the star rating.
        "p",                             # Look for a <p> HTML element.
        class_="star-rating"             # Select the paragraph containing the rating class.
    )["class"]                            # Retrieve the list of HTML classes.

    rating_word = rating_classes[1]       # Take the second class, such as "Three" or "Five".

    rating = rating_map[rating_word]      # Convert the rating word into a numeric value.

    titles.append(title)                  # Add the extracted title to the titles list.
    prices.append(price)                  # Add the extracted price to the prices list.
    ratings.append(rating)                # Add the numeric rating to the ratings list.
    availability_list.append(availability) # Add availability information to its list.

In [ ]:
print(rating_map)

{'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}


In [ ]:
# ------------------------------------------------------------
# STEP 9: CREATE A PANDAS DATAFRAME
# ------------------------------------------------------------

df = pd.DataFrame({                       # Create a DataFrame from the four collected lists.
    "Book_Title": titles,                 # Create the Book_Title column.
    "Price": prices,                      # Create the Price column.
    "Rating": ratings,                    # Create the Rating column.
    "Availability": availability_list     # Create the Availability column.
})


# ------------------------------------------------------------
# STEP 10: VIEW RAW SCRAPED DATA
# ------------------------------------------------------------

print("\nRaw Scraped Data:")              # Print a heading for the raw data.
print(df.head())                          # Display the first five scraped records.


Raw Scraped Data:
                              Book_Title   Price  Rating Availability
0                   A Light in the Attic  £51.77       3     In stock
1                     Tipping the Velvet  £53.74       1     In stock
2                             Soumission  £50.10       1     In stock
3                          Sharp Objects  £47.82       4     In stock
4  Sapiens: A Brief History of Humankind  £54.23       5     In stock


In [ ]:

# ------------------------------------------------------------
# STEP 11: CLEAN THE PRICE COLUMN
# ------------------------------------------------------------

df["Price"] = df["Price"].str.replace(    # Remove unwanted characters from the Price column.
    r"[^\d.]",                            # Keep only digits and the decimal point.
    "",                                  # Replace unwanted characters with nothing.
    regex=True                            # Treat the search pattern as a regular expression.
)

df["Price"] = pd.to_numeric(              # Convert cleaned price text into numeric values.
    df["Price"],                          # Specify the Price column to convert.
    errors="coerce"                       # Convert invalid values into NaN instead of crashing.
)


# ------------------------------------------------------------
# STEP 12: INSPECT THE CLEANED DATA
# ------------------------------------------------------------

print("\nFirst Five Cleaned Records:")    # Print a heading.
print(df.head())                          # Display the first five cleaned records.

print("\nDataset Shape:")                 # Print a heading for dataset size.
print(df.shape)                           # Display the number of rows and columns.

print("\nColumn Data Types:")             # Print a heading for data types.
print(df.dtypes)                          # Display the data type of every column.

print("\nMissing Values:")                # Print a heading for missing values.
print(df.isnull().sum())                  # Count missing values in every column.

print("\nDuplicates:")                    # Print a heading for duplicate records.
print(df.duplicated().sum())              # Count fully duplicated rows.




First Five Cleaned Records:
                              Book_Title  Price  Rating Availability
0                   A Light in the Attic  51.77       3     In stock
1                     Tipping the Velvet  53.74       1     In stock
2                             Soumission  50.10       1     In stock
3                          Sharp Objects  47.82       4     In stock
4  Sapiens: A Brief History of Humankind  54.23       5     In stock

Dataset Shape:
(20, 4)

Column Data Types:
Book_Title       object
Price           float64
Rating            int64
Availability     object
dtype: object

Missing Values:
Book_Title      0
Price           0
Rating          0
Availability    0
dtype: int64

Duplicates:
0


In [ ]:
# ------------------------------------------------------------
# STEP 13: SUMMARY STATISTICS
# ------------------------------------------------------------

print("\nSummary Statistics:")            # Print a heading for descriptive statistics.
print(df.describe())                      # Display statistics for numeric columns.


Summary Statistics:
           Price     Rating
count  20.000000  20.000000
mean   38.048500   2.850000
std    15.135231   1.565248
min    13.990000   1.000000
25%    22.637500   1.000000
50%    41.380000   3.000000
75%    51.865000   4.000000
max    57.250000   5.000000


In [ ]:
# ------------------------------------------------------------
# STEP 14: CALCULATE AVERAGE PRICE
# ------------------------------------------------------------

average_price = df["Price"].mean()         # Calculate the mean price of all scraped books.

print(                                    # Start a print statement.
    "\nAverage Price:",                   # Display a label.
    round(average_price, 2)               # Round the average price to two decimal places.
)


Average Price: 38.05


In [ ]:
# ------------------------------------------------------------
# STEP 15: CALCULATE MEDIAN PRICE
# ------------------------------------------------------------

median_price = df["Price"].median()        # Calculate the median book price.

print(                                    # Start a print statement.
    "Median Price:",                      # Display a label.
    round(median_price, 2)                # Round the median price to two decimal places.
)

Median Price: 41.38


In [ ]:
# ------------------------------------------------------------
# STEP 16: CALCULATE AVERAGE RATING
# ------------------------------------------------------------

average_rating = df["Rating"].mean()       # Calculate the average star rating.

print(                                    # Start a print statement.
    "Average Rating:",                    # Display a label.
    round(average_rating, 2)              # Round the average rating to two decimal places.
)


Average Rating: 2.85


In [ ]:
# ------------------------------------------------------------
# STEP 17: RATING DISTRIBUTION
# ------------------------------------------------------------

rating_distribution = (                   # Create a variable to store rating counts.
    df["Rating"]                          # Select the Rating column.
    .value_counts()                       # Count how many books have each rating.
    .sort_index()                         # Sort ratings from 1 star to 5 stars.
)

print("\nRating Distribution:")           # Print a heading for rating distribution.
print(rating_distribution)                # Display the number of books in each rating group.


Rating Distribution:
Rating
1    6
2    3
3    3
4    4
5    4
Name: count, dtype: int64


In [ ]:
# ------------------------------------------------------------
# STEP 18: FIND THE MOST EXPENSIVE BOOK
# ------------------------------------------------------------

most_expensive = df.loc[                  # Select the full row of the most expensive book.
    df["Price"].idxmax()                  # Find the row index containing the maximum price.
]

print("\nMost Expensive Book:")           # Print a heading.
print(most_expensive)                     # Display information about the most expensive book.


Most Expensive Book:
Book_Title      Our Band Could Be Your Life: Scenes from the A...
Price                                                       57.25
Rating                                                          3
Availability                                             In stock
Name: 15, dtype: object


In [ ]:
# ------------------------------------------------------------
# STEP 19: FIND THE LEAST EXPENSIVE BOOK
# ------------------------------------------------------------

least_expensive = df.loc[                 # Select the full row of the least expensive book.
    df["Price"].idxmin()                  # Find the row index containing the minimum price.
]

print("\nLeast Expensive Book:")          # Print a heading.
print(least_expensive)                    # Display information about the least expensive book.


Least Expensive Book:
Book_Title      Starving Hearts (Triangular Trade Trilogy, #1)
Price                                                    13.99
Rating                                                       2
Availability                                          In stock
Name: 10, dtype: object


In [ ]:
# ------------------------------------------------------------
# STEP 20: FIND HIGH-RATED PRODUCTS
# ------------------------------------------------------------

high_rated = df[                          # Create a filtered DataFrame.
    df["Rating"] >= 4                     # Keep only books with ratings of 4 or 5 stars.
]

print("\nHighly Rated Books:")            # Print a heading.
print(high_rated)                         # Display all highly rated books.


Highly Rated Books:
                                           Book_Title  Price  Rating  \
3                                       Sharp Objects  47.82       4   
4               Sapiens: A Brief History of Humankind  54.23       5   
6   The Dirty Little Secrets of Getting Your Dream...  33.34       4   
8   The Boys in the Boat: Nine Americans and Their...  22.60       4   
11                              Shakespeare's Sonnets  20.66       4   
12                                        Set Me Free  17.46       5   
13  Scott Pilgrim's Precious Little Life (Scott Pi...  52.29       5   
14                          Rip it Up and Start Again  35.02       5   

   Availability  
3      In stock  
4      In stock  
6      In stock  
8      In stock  
11     In stock  
12     In stock  
13     In stock  
14     In stock  


In [ ]:
# ------------------------------------------------------------
# STEP 21: FIND LOWER-PRICED AND HIGH-RATED BOOKS
# ------------------------------------------------------------

opportunities = df[                       # Create another filtered DataFrame.
    (df["Rating"] >= 4) &                 # Require a rating of at least 4 stars.
    (df["Price"] <= median_price)         # Require price to be at or below the median price.
]

print("\nPotential Opportunities:")       # Print a heading.
print(opportunities)                      # Display potentially attractive books.


Potential Opportunities:
                                           Book_Title  Price  Rating  \
6   The Dirty Little Secrets of Getting Your Dream...  33.34       4   
8   The Boys in the Boat: Nine Americans and Their...  22.60       4   
11                              Shakespeare's Sonnets  20.66       4   
12                                        Set Me Free  17.46       5   
14                          Rip it Up and Start Again  35.02       5   

   Availability  
6      In stock  
8      In stock  
11     In stock  
12     In stock  
14     In stock  


In [ ]:

# ------------------------------------------------------------
# STEP 22: CALCULATE AVERAGE PRICE BY RATING
# ------------------------------------------------------------

price_by_rating = (                       # Create a summary table.
    df.groupby("Rating")["Price"]          # Group book prices according to star rating.
    .mean()                               # Calculate the average price for each rating group.
    .round(2)                             # Round values to two decimal places.
)

print("\nAverage Price by Rating:")       # Print a heading.
print(price_by_rating)                    # Display average price for each rating.


Average Price by Rating:
Rating
1    40.02
2    36.83
3    42.32
4    31.10
5    39.75
Name: Price, dtype: float64


In [ ]:
# ------------------------------------------------------------
# STEP 23: SAVE THE SCRAPED DATA
# ------------------------------------------------------------

df.to_csv(                                # Export the DataFrame to a CSV file.
    "bookwise_competitor_data.csv",       # Specify the output filename.
    index=False                           # Do not save Pandas row numbers as a column.
)

print(                                    # Start a final print statement.
    "\nDataset successfully saved as "    # Display confirmation text.
    "'bookwise_competitor_data.csv'."     # Display the output filename.
)


Dataset successfully saved as 'bookwise_competitor_data.csv'.


In [ ]:
# ------------------------------------------------------------
# STEP 24: FINAL BUSINESS SUMMARY
# ------------------------------------------------------------

print("\n--- BUSINESS SUMMARY ---")       # Print a heading for the final summary.

print("Books Collected:", len(df))         # Display total number of collected books.

print(                                    # Start a print statement.
    "Average Listed Price:",              # Display a label.
    round(df["Price"].mean(), 2)          # Display the average price.
)

print(                                    # Start a print statement.
    "Median Listed Price:",               # Display a label.
    round(df["Price"].median(), 2)        # Display the median price.
)

print(                                    # Start a print statement.
    "Books Rated 4 or Higher:",           # Display a label.
    len(high_rated)                       # Count books with ratings of 4 or 5.
)

print(                                    # Start a print statement.
    "Potential Opportunities:",           # Display a label.
    len(opportunities)                    # Count high-rated books at or below median price.
)


--- BUSINESS SUMMARY ---
Books Collected: 20
Average Listed Price: 38.05
Median Listed Price: 41.38
Books Rated 4 or Higher: 8
Potential Opportunities: 5


APIs

In [ ]:
# ============================================================
# BEGINNER TUTORIAL: USING API DATA WITH PYTHON
# ============================================================

# Import the requests library.
# This library is used to send requests to websites and APIs.
import requests

# Import the pandas library.
# Pandas is used to organize and analyze data in table form.
import pandas as pd


# ============================================================
# STEP 1: DEFINE THE API URL
# ============================================================

# Store the API endpoint inside a variable called url.
# An endpoint is the specific API address from which we request data.
# JSONPlaceholder is a free practice API that does not require an API key.
url = "https://jsonplaceholder.typicode.com/users"


# ============================================================
# STEP 2: SEND A GET REQUEST TO THE API
# ============================================================

# Send a GET request to the API.
# GET is used when we want to read or download data.
response = requests.get(url)


# ============================================================
# STEP 3: CHECK THE HTTP STATUS CODE
# ============================================================

# Print the status code returned by the API server.
# Status code 200 means that the request was successful.
print("Status Code:", response.status_code)


# ============================================================
# STEP 4: CHECK WHETHER THE REQUEST WAS SUCCESSFUL
# ============================================================

# Check whether the API returned status code 200.
if response.status_code == 200:

    # Display a success message if the request was successful.
    print("API request was successful.")

else:

    # Display an error message if the API request failed.
    print("API request failed.")

    # Stop the program because there is no valid data to process.
    exit()

Status Code: 200
API request was successful.


In [ ]:
# ============================================================
# STEP 5: CONVERT THE API RESPONSE INTO JSON
# ============================================================

# Convert the API response from JSON format into Python data.
# JSON data usually becomes a Python list or dictionary.
data = response.json()


# ============================================================
# STEP 6: CHECK THE TYPE OF DATA RECEIVED
# ============================================================

# Print the Python data type of the returned API data.
print("\nType of API Data:")

# Display the data type.
print(type(data))


Type of API Data:
<class 'list'>


In [ ]:

# ============================================================
# STEP 7: CHECK HOW MANY RECORDS WERE RECEIVED
# ============================================================

# Count the number of records returned by the API.
number_of_records = len(data)

# Display the total number of records.
print("\nTotal Records Received:", number_of_records)


Total Records Received: 10


In [ ]:

# ============================================================
# STEP 8: VIEW THE FIRST RECORD
# ============================================================

# Get the first record from the API data.
# Python starts counting from 0, so data[0] means the first record.
first_user = data[0]

# Display a heading.
print("\nFirst Record:")

# Print the complete first user record.
print(first_user)


First Record:
{'id': 1, 'name': 'Leanne Graham', 'username': 'Bret', 'email': 'Sincere@april.biz', 'address': {'street': 'Kulas Light', 'suite': 'Apt. 556', 'city': 'Gwenborough', 'zipcode': '92998-3874', 'geo': {'lat': '-37.3159', 'lng': '81.1496'}}, 'phone': '1-770-736-8031 x56442', 'website': 'hildegard.org', 'company': {'name': 'Romaguera-Crona', 'catchPhrase': 'Multi-layered client-server neural-net', 'bs': 'harness real-time e-markets'}}


In [ ]:
# ============================================================
# STEP 9: ACCESS INDIVIDUAL VALUES FROM THE FIRST RECORD
# ============================================================

# Get the ID of the first user.
user_id = first_user["id"]

# Get the name of the first user.
user_name = first_user["name"]

# Get the username of the first user.
username = first_user["username"]

# Get the email address of the first user.
email = first_user["email"]


# Display a heading.
print("\nSelected Information from First Record:")

# Display the user's ID.
print("ID:", user_id)

# Display the user's name.
print("Name:", user_name)

# Display the username.
print("Username:", username)

# Display the user's email address.
print("Email:", email)


Selected Information from First Record:
ID: 1
Name: Leanne Graham
Username: Bret
Email: Sincere@april.biz


In [ ]:
# ============================================================
# STEP 10: LOOP THROUGH ALL API RECORDS
# ============================================================

# Display a heading before printing all users.
print("\nAll Users:")

# Start a loop that processes every user in the API data.
for user in data:

    # Get the ID of the current user.
    current_id = user["id"]

    # Get the name of the current user.
    current_name = user["name"]

    # Get the email address of the current user.
    current_email = user["email"]

    # Display the current user's information.
    print(current_id, "-", current_name, "-", current_email)


All Users:
1 - Leanne Graham - Sincere@april.biz
2 - Ervin Howell - Shanna@melissa.tv
3 - Clementine Bauch - Nathan@yesenia.net
4 - Patricia Lebsack - Julianne.OConner@kory.org
5 - Chelsey Dietrich - Lucio_Hettinger@annie.ca
6 - Mrs. Dennis Schulist - Karley_Dach@jasper.info
7 - Kurtis Weissnat - Telly.Hoeger@billy.biz
8 - Nicholas Runolfsdottir V - Sherwood@rosamond.me
9 - Glenna Reichert - Chaim_McDermott@dana.io
10 - Clementina DuBuque - Rey.Padberg@karina.biz


In [ ]:
# ============================================================
# STEP 11: CONVERT JSON DATA INTO A PANDAS DATAFRAME
# ============================================================

# Convert the JSON data into a Pandas DataFrame.
# A DataFrame stores data in rows and columns like Excel.
df = pd.DataFrame(data)


# ============================================================
# STEP 12: VIEW THE COMPLETE DATAFRAME
# ============================================================

# Display a heading.
print("\nComplete DataFrame:")

# Print the DataFrame.
print(df)


Complete DataFrame:
   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

                                             address                  phone  \
0  {'street': 'Kula

In [ ]:
# ============================================================
# STEP 13: VIEW THE FIRST FIVE ROWS
# ============================================================

# Display a heading.
print("\nFirst Five Rows:")

# Display the first five rows of the DataFrame.
print(df.head())


First Five Rows:
   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                                             address                  phone  \
0  {'street': 'Kulas Light', 'suite': 'Apt. 556',...  1-770-736-8031 x56442   
1  {'street': 'Victor Plains', 'suite': 'Suite 87...    010-692-6593 x09125   
2  {'street': 'Douglas Extension', 'suite': 'Suit...         1-463-123-4447   
3  {'street': 'Hoeger Mall', 'suite': 'Apt. 692',...      493-170-9623 x156   
4  {'street': 'Skiles Walks', 'suite': 'Suite 351...          (254)954-1289   

         website                                            company  
0  hildegard.org  {'name': 'Romaguera-Crona', 

In [ ]:
# ============================================================
# STEP 14: CHECK THE COLUMN NAMES
# ============================================================

# Display a heading.
print("\nColumn Names:")

# Display all available column names.
print(df.columns)


Column Names:
Index(['id', 'name', 'username', 'email', 'address', 'phone', 'website',
       'company'],
      dtype='object')


In [ ]:
# ============================================================
# STEP 15: SELECT ONLY IMPORTANT COLUMNS
# ============================================================

# Select only the columns that we want to keep.
# This removes unnecessary nested columns such as address and company.
df = df[
    [
        "id",
        "name",
        "username",
        "email",
        "phone",
        "website"
    ]
]


# ============================================================
# STEP 16: DISPLAY THE CLEAN DATA
# ============================================================

# Display a heading.
print("\nSelected API Data:")

# Display the cleaned DataFrame.
print(df)


Selected API Data:
   id                      name          username                      email  \
0   1             Leanne Graham              Bret          Sincere@april.biz   
1   2              Ervin Howell         Antonette          Shanna@melissa.tv   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   
4   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
5   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
6   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
7   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
8   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
9  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   

                   phone        website  
0  1-770-736-8031 x56442  hildegard.org  
1    010-692-65

In [ ]:
# ============================================================
# STEP 17: CHECK THE SIZE OF THE DATASET
# ============================================================

# Get the number of rows in the DataFrame.
number_of_rows = df.shape[0]

# Get the number of columns in the DataFrame.
number_of_columns = df.shape[1]

# Display the total number of rows.
print("\nNumber of Rows:", number_of_rows)

# Display the total number of columns.
print("Number of Columns:", number_of_columns)


Number of Rows: 10
Number of Columns: 6


In [ ]:
# ============================================================
# STEP 18: CHECK DATA TYPES
# ============================================================

# Display a heading.
print("\nData Types:")

# Display the data type of each column.
print(df.dtypes)


# ============================================================
# STEP 19: CHECK FOR MISSING VALUES
# ============================================================

# Display a heading.
print("\nMissing Values:")

# Count missing values in every column.
print(df.isnull().sum())


Data Types:
id           int64
name        object
username    object
email       object
phone       object
website     object
dtype: object

Missing Values:
id          0
name        0
username    0
email       0
phone       0
website     0
dtype: int64


In [ ]:
# ============================================================
# STEP 20: CHECK FOR DUPLICATE RECORDS
# ============================================================

# Count the number of duplicate rows.
duplicate_records = df.duplicated().sum()

# Display the number of duplicate records.
print("\nDuplicate Records:", duplicate_records)


Duplicate Records: 0


In [ ]:

# ============================================================
# STEP 21: REMOVE DUPLICATES
# ============================================================

# Remove duplicate rows if any duplicates exist.
df = df.drop_duplicates()


# ============================================================
# STEP 22: SORT THE DATA
# ============================================================

# Sort the users alphabetically according to their names.
df = df.sort_values(
    by="name"
)


# ============================================================
# STEP 23: RESET THE DATAFRAME INDEX
# ============================================================

# Reset the row numbers after sorting.
# drop=True prevents the old index from becoming a new column.
df = df.reset_index(
    drop=True
)


# ============================================================
# STEP 24: DISPLAY THE FINAL CLEAN DATASET
# ============================================================

# Display a heading.
print("\nFinal Clean API Dataset:")

# Display the final cleaned and sorted DataFrame.
print(df)


Final Clean API Dataset:
   id                      name          username                      email  \
0   5          Chelsey Dietrich            Kamren   Lucio_Hettinger@annie.ca   
1  10        Clementina DuBuque    Moriah.Stanton     Rey.Padberg@karina.biz   
2   3          Clementine Bauch          Samantha         Nathan@yesenia.net   
3   2              Ervin Howell         Antonette          Shanna@melissa.tv   
4   9           Glenna Reichert          Delphine    Chaim_McDermott@dana.io   
5   7           Kurtis Weissnat      Elwyn.Skiles     Telly.Hoeger@billy.biz   
6   1             Leanne Graham              Bret          Sincere@april.biz   
7   6      Mrs. Dennis Schulist  Leopoldo_Corkery    Karley_Dach@jasper.info   
8   8  Nicholas Runolfsdottir V     Maxime_Nienow       Sherwood@rosamond.me   
9   4          Patricia Lebsack          Karianne  Julianne.OConner@kory.org   

                   phone        website  
0          (254)954-1289   demarco.info  
1        

In [ ]:

# ============================================================
# STEP 25: SAVE THE API DATA AS A CSV FILE
# ============================================================

# Store the output file name in a variable.
file_name = "users_api_data.csv"

# Save the DataFrame as a CSV file.
# index=False prevents Pandas from saving row numbers as a separate column.
df.to_csv(
    file_name,
    index=False
)


# ============================================================
# STEP 26: CONFIRM THAT THE FILE WAS SAVED
# ============================================================

# Display a confirmation message.
print("\nData successfully saved as:", file_name)


Data successfully saved as: users_api_data.csv


In [ ]:
# ============================================================
# FINAL SUMMARY
# ============================================================

# Display a heading.
print("\nAPI DATA COLLECTION COMPLETED")

# Display the basic workflow used in this program.
print("API URL -> GET Request -> JSON -> DataFrame -> Cleaning -> CSV")


API DATA COLLECTION COMPLETED
API URL -> GET Request -> JSON -> DataFrame -> Cleaning -> CSV
